# 混合精度訓練 (Mixed Precision Training)
:label:`sec_mixed_precision`

混合精度訓練是現代深度學習中最重要的性能優化技術之一。通過在訓練過程中同時使用 FP32（單精度浮點）和 FP16（半精度浮點）數據類型，我們可以：

- **加速訓練 2-3 倍**：FP16 計算速度更快
- **減少記憶體使用 40-50%**：可以使用更大的批次大小
- **保持模型準確度**：關鍵操作仍使用 FP32

本章將深入探討混合精度訓練的原理、實現和最佳實踐。

## 目錄
1. [浮點數精度基礎](#1-浮點數精度基礎)
2. [為什麼需要混合精度？](#2-為什麼需要混合精度)
3. [PyTorch AMP 實戰](#3-pytorch-amp-實戰)
4. [梯度縮放技術](#4-梯度縮放技術)
5. [BF16 vs FP16](#5-bf16-vs-fp16)
6. [常見問題與解決方案](#6-常見問題與解決方案)
7. [性能基準測試](#7-性能基準測試)

In [ ]:
# 導入必要的庫
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 檢查 CUDA 可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")

## 1. 浮點數精度基礎

### 1.1 不同精度格式對比

| 格式 | 總位數 | 符號位 | 指數位 | 尾數位 | 數值範圍 | 精度 |
|------|--------|--------|--------|--------|----------|------|
| FP32 | 32 | 1 | 8 | 23 | ±3.4e38 | ~7 位十進制 |
| FP16 | 16 | 1 | 5 | 10 | ±65504 | ~3 位十進制 |
| BF16 | 16 | 1 | 8 | 7 | ±3.4e38 | ~2 位十進制 |

### 1.2 視覺化精度差異

In [ ]:
def visualize_precision():
    """視覺化不同精度格式的表示能力"""
    # 測試數值
    test_values = [0.1, 0.01, 0.001, 0.0001, 1e-5, 1e-6, 1e-7, 1e-8]
    
    results = {'value': [], 'fp32': [], 'fp16': [], 'bf16': []}
    
    for val in test_values:
        fp32_val = torch.tensor(val, dtype=torch.float32)
        fp16_val = fp32_val.half()
        bf16_val = fp32_val.bfloat16()
        
        results['value'].append(val)
        results['fp32'].append(fp32_val.item())
        results['fp16'].append(fp16_val.item())
        results['bf16'].append(bf16_val.item())
    
    # 計算誤差
    print("精度比較（絕對誤差）：")
    print("-" * 70)
    print(f"{'原始值':<15} {'FP32':<15} {'FP16 誤差':<15} {'BF16 誤差':<15}")
    print("-" * 70)
    
    for i, val in enumerate(test_values):
        fp16_error = abs(results['fp16'][i] - val)
        bf16_error = abs(results['bf16'][i] - val)
        print(f"{val:<15.2e} {results['fp32'][i]:<15.2e} {fp16_error:<15.2e} {bf16_error:<15.2e}")

visualize_precision()

## 2. 為什麼需要混合精度？

### 2.1 FP16 的優勢

1. **硬體加速**：現代 GPU（Volta, Turing, Ampere）有專門的 Tensor Cores 用於 FP16 計算
2. **記憶體節省**：FP16 只需 FP32 一半的記憶體
3. **帶寬優化**：記憶體傳輸時間減半

### 2.2 FP16 的挑戰

1. **數值範圍有限**：容易 overflow/underflow
2. **精度損失**：小梯度可能變為零
3. **累積誤差**：長序列操作可能放大誤差

In [ ]:
# 演示 FP16 的數值問題
def demonstrate_fp16_issues():
    print("=== FP16 數值範圍測試 ===")
    
    # 1. Overflow 問題
    large_num = torch.tensor([60000.0], dtype=torch.float32)
    print(f"\n1. Overflow 測試：")
    print(f"   FP32: {large_num.item()}")
    print(f"   FP16: {large_num.half().item()}")
    print(f"   超過 FP16 最大值 (65504) 會變為 inf")
    
    # 2. Underflow 問題
    small_num = torch.tensor([1e-8], dtype=torch.float32)
    print(f"\n2. Underflow 測試：")
    print(f"   FP32: {small_num.item():.2e}")
    print(f"   FP16: {small_num.half().item():.2e}")
    print(f"   小於 FP16 最小值會變為 0")
    
    # 3. 精度損失
    nums = torch.randn(1000, dtype=torch.float32) * 0.001
    fp32_sum = nums.sum()
    fp16_sum = nums.half().sum().float()
    print(f"\n3. 累積誤差測試（1000個小數相加）：")
    print(f"   FP32 結果: {fp32_sum.item():.6f}")
    print(f"   FP16 結果: {fp16_sum.item():.6f}")
    print(f"   相對誤差: {abs(fp32_sum - fp16_sum) / abs(fp32_sum) * 100:.2f}%")

demonstrate_fp16_issues()

## 3. PyTorch AMP 實戰

PyTorch 的自動混合精度（Automatic Mixed Precision, AMP）讓我們可以輕鬆使用混合精度訓練，同時自動處理數值穩定性問題。

### 3.1 基本使用範例

In [ ]:
# 定義一個簡單的模型
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 傳統訓練（FP32）
def train_fp32(model, data, target, optimizer, criterion):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    return loss.item()

# 混合精度訓練（AMP）
def train_amp(model, data, target, optimizer, criterion, scaler):
    optimizer.zero_grad()
    
    # 使用 autocast 自動選擇精度
    with autocast():
        output = model(data)
        loss = criterion(output, target)
    
    # 使用 GradScaler 縮放梯度
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    return loss.item()

print("訓練函數定義完成！")

### 3.2 完整訓練範例

In [ ]:
def compare_training_modes():
    """比較 FP32 和 AMP 訓練的性能"""
    
    # 準備數據
    batch_size = 64
    dummy_data = torch.randn(batch_size, 3, 32, 32, device=device)
    dummy_target = torch.randint(0, 10, (batch_size,), device=device)
    
    criterion = nn.CrossEntropyLoss()
    num_iterations = 100
    
    results = {}
    
    # 測試 FP32
    print("\n=== 測試 FP32 訓練 ===")
    model_fp32 = SimpleNet().to(device)
    optimizer_fp32 = optim.Adam(model_fp32.parameters(), lr=0.001)
    
    start_time = time.time()
    for i in range(num_iterations):
        loss = train_fp32(model_fp32, dummy_data, dummy_target, optimizer_fp32, criterion)
        if i % 20 == 0:
            print(f"  Iter {i}: Loss = {loss:.4f}")
    fp32_time = time.time() - start_time
    
    if torch.cuda.is_available():
        fp32_memory = torch.cuda.max_memory_allocated() / 1024**2
        torch.cuda.reset_peak_memory_stats()
    
    # 測試 AMP
    print("\n=== 測試 AMP 訓練 ===")
    model_amp = SimpleNet().to(device)
    optimizer_amp = optim.Adam(model_amp.parameters(), lr=0.001)
    scaler = GradScaler()
    
    start_time = time.time()
    for i in range(num_iterations):
        loss = train_amp(model_amp, dummy_data, dummy_target, optimizer_amp, criterion, scaler)
        if i % 20 == 0:
            print(f"  Iter {i}: Loss = {loss:.4f}")
    amp_time = time.time() - start_time
    
    if torch.cuda.is_available():
        amp_memory = torch.cuda.max_memory_allocated() / 1024**2
    
    # 總結
    print("\n" + "="*60)
    print("性能比較結果")
    print("="*60)
    print(f"訓練時間:")
    print(f"  FP32: {fp32_time:.2f} 秒")
    print(f"  AMP:  {amp_time:.2f} 秒")
    print(f"  加速比: {fp32_time/amp_time:.2f}x")
    
    if torch.cuda.is_available():
        print(f"\n記憶體使用:")
        print(f"  FP32: {fp32_memory:.2f} MB")
        print(f"  AMP:  {amp_memory:.2f} MB")
        print(f"  節省: {(1 - amp_memory/fp32_memory)*100:.1f}%")

if torch.cuda.is_available():
    compare_training_modes()
else:
    print("需要 GPU 才能運行此測試")

## 4. 梯度縮放技術

梯度縮放（Gradient Scaling）是混合精度訓練的核心技術，用於解決 FP16 的 underflow 問題。

### 4.1 工作原理

```python
# 1. 計算 loss 時放大
scaled_loss = loss * scale_factor  # 例如 scale_factor = 2^16

# 2. 反向傳播（梯度也被放大）
scaled_loss.backward()

# 3. 更新權重前縮小梯度
for param in model.parameters():
    param.grad /= scale_factor
```

### 4.2 動態梯度縮放

In [ ]:
# 演示 GradScaler 的動態縮放
def demonstrate_grad_scaler():
    print("=== GradScaler 動態縮放演示 ===")
    
    scaler = GradScaler()
    model = SimpleNet().to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    
    dummy_data = torch.randn(16, 3, 32, 32, device=device)
    dummy_target = torch.randint(0, 10, (16,), device=device)
    criterion = nn.CrossEntropyLoss()
    
    print(f"\n初始縮放因子: {scaler.get_scale()}")
    
    for i in range(10):
        optimizer.zero_grad()
        
        with autocast():
            output = model(dummy_data)
            loss = criterion(output, dummy_target)
        
        scaler.scale(loss).backward()
        
        # 檢查梯度是否包含 inf/nan
        scaler.step(optimizer)
        scaler.update()
        
        if i % 2 == 0:
            print(f"Iteration {i}: 縮放因子 = {scaler.get_scale():.0f}")
    
    print("\n說明: GradScaler 會自動調整縮放因子以避免溢出")

if torch.cuda.is_available():
    demonstrate_grad_scaler()
else:
    print("需要 GPU 才能運行此演示")

## 5. BF16 vs FP16

BFloat16 (BF16) 是 Google 提出的另一種 16 位格式，在某些情況下比 FP16 更適合深度學習。

### 5.1 主要差異

- **FP16**: 更高精度（10 位尾數），較小範圍（5 位指數）
- **BF16**: 較低精度（7 位尾數），與 FP32 相同範圍（8 位指數）

### 5.2 使用場景

- **FP16**: 推理、小模型訓練
- **BF16**: 大模型訓練（如 GPT, BERT），不需要梯度縮放

In [ ]:
# 比較 FP16 和 BF16
def compare_fp16_bf16():
    print("=== FP16 vs BF16 比較 ===")
    
    # 測試各種數值
    test_cases = [
        ("正常數值", 1.234),
        ("大數值", 12345.6),
        ("小數值", 0.00012345),
        ("極小值", 1e-7),
    ]
    
    print(f"\n{'類型':<10} {'原始值':<15} {'FP16':<15} {'BF16':<15} {'FP16誤差':<15} {'BF16誤差':<15}")
    print("-" * 85)
    
    for name, val in test_cases:
        fp32 = torch.tensor(val)
        fp16 = fp32.half()
        
        # 檢查是否支持 bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            bf16 = fp32.bfloat16()
            bf16_val = bf16.float().item()
            bf16_err = abs(bf16_val - val)
        else:
            bf16_val = "不支持"
            bf16_err = "-"
        
        fp16_val = fp16.float().item()
        fp16_err = abs(fp16_val - val)
        
        print(f"{name:<10} {val:<15.8f} {fp16_val:<15.8f} {str(bf16_val):<15} "
              f"{fp16_err:<15.2e} {str(bf16_err):<15}")
    
    # 硬體支援檢查
    print("\n硬體支援:")
    print(f"  FP16: {'✓' if torch.cuda.is_available() else '✗'}")
    if torch.cuda.is_available():
        print(f"  BF16: {'✓' if torch.cuda.is_bf16_supported() else '✗ (需要 Ampere+ GPU)'}")

compare_fp16_bf16()

## 6. 常見問題與解決方案

### 6.1 Loss 變為 NaN

**原因**: 梯度爆炸或數值溢出

**解決方案**:

In [ ]:
# 方案 1: 梯度裁剪
from torch.nn.utils import clip_grad_norm_

def train_with_grad_clip(model, data, target, optimizer, criterion, scaler, max_norm=1.0):
    optimizer.zero_grad()
    
    with autocast():
        output = model(data)
        loss = criterion(output, target)
    
    scaler.scale(loss).backward()
    
    # 梯度裁剪
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm)
    
    scaler.step(optimizer)
    scaler.update()
    
    return loss.item()

# 方案 2: 降低學習率
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # 從 1e-3 降到 1e-4

# 方案 3: 使用 BF16（如果硬體支援）
# with autocast(dtype=torch.bfloat16):
#     output = model(data)

print("常見問題解決方案已定義")

### 6.2 某些操作不支持 FP16

In [ ]:
# 手動控制精度
def custom_autocast_example(x):
    with autocast():
        # 大部分操作使用 FP16
        x = nn.functional.relu(x)
        x = nn.functional.conv2d(x, weight)
        
        # 某些操作強制使用 FP32
        with autocast(enabled=False):
            x = x.float()  # 轉回 FP32
            x = torch.norm(x)  # 使用 FP32 計算
            x = x.half()  # 轉回 FP16
    
    return x

print("自定義 autocast 範例已定義")

## 7. 性能基準測試

讓我們在實際的 ResNet 模型上進行完整的性能測試。

In [ ]:
def comprehensive_benchmark():
    """完整的性能基準測試"""
    print("\n" + "="*70)
    print("ResNet-50 訓練性能基準測試")
    print("="*70)
    
    if not torch.cuda.is_available():
        print("需要 GPU 才能運行基準測試")
        return
    
    # 準備模型和數據
    model = models.resnet50(pretrained=False).to(device)
    criterion = nn.CrossEntropyLoss()
    batch_size = 64
    num_iterations = 50
    
    # 生成假數據
    data = torch.randn(batch_size, 3, 224, 224, device=device)
    target = torch.randint(0, 1000, (batch_size,), device=device)
    
    results = {}
    
    # 測試配置
    configs = [
        ('FP32 Baseline', False, False),
        ('AMP (FP16)', True, False),
    ]
    
    # 添加 BF16 測試（如果支援）
    if torch.cuda.is_bf16_supported():
        configs.append(('AMP (BF16)', True, True))
    
    for name, use_amp, use_bf16 in configs:
        print(f"\n測試: {name}")
        print("-" * 70)
        
        # 重置模型
        model = models.resnet50(pretrained=False).to(device)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        scaler = GradScaler(enabled=use_amp)
        
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        
        # 預熱
        for _ in range(5):
            optimizer.zero_grad()
            if use_amp:
                dtype = torch.bfloat16 if use_bf16 else torch.float16
                with autocast(dtype=dtype):
                    output = model(data)
                    loss = criterion(output, target)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
        
        torch.cuda.synchronize()
        
        # 正式測試
        start_time = time.time()
        
        for i in range(num_iterations):
            optimizer.zero_grad()
            
            if use_amp:
                dtype = torch.bfloat16 if use_bf16 else torch.float16
                with autocast(dtype=dtype):
                    output = model(data)
                    loss = criterion(output, target)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
        
        torch.cuda.synchronize()
        elapsed = time.time() - start_time
        
        # 收集結果
        throughput = (num_iterations * batch_size) / elapsed
        memory = torch.cuda.max_memory_allocated() / 1024**3  # GB
        
        results[name] = {
            'time': elapsed,
            'throughput': throughput,
            'memory': memory
        }
        
        print(f"  總時間: {elapsed:.2f} 秒")
        print(f"  吞吐量: {throughput:.1f} images/sec")
        print(f"  記憶體: {memory:.2f} GB")
    
    # 總結比較
    print("\n" + "="*70)
    print("性能提升總結")
    print("="*70)
    
    baseline = results['FP32 Baseline']
    
    for name, metrics in results.items():
        if name == 'FP32 Baseline':
            continue
        
        speedup = baseline['time'] / metrics['time']
        memory_saving = (1 - metrics['memory'] / baseline['memory']) * 100
        
        print(f"\n{name}:")
        print(f"  速度提升: {speedup:.2f}x")
        print(f"  記憶體節省: {memory_saving:.1f}%")
        print(f"  吞吐量提升: {(metrics['throughput'] / baseline['throughput'] - 1) * 100:.1f}%")

comprehensive_benchmark()

## 總結

### 關鍵要點

1. **混合精度訓練可以帶來 2-3 倍加速**，同時減少 40-50% 記憶體使用
2. **使用 PyTorch AMP 非常簡單**：只需 `autocast()` + `GradScaler()`
3. **梯度縮放是關鍵**：解決 FP16 的數值範圍問題
4. **選擇合適的精度格式**：
   - FP16: 推理、小模型
   - BF16: 大模型訓練（需要 Ampere+ GPU）
5. **注意數值穩定性**：使用梯度裁剪、適當的學習率

### 最佳實踐

```python
# 推薦的訓練模板
model = YourModel().to(device)
optimizer = torch.optim.Adam(model.parameters())
scaler = GradScaler()

for data, target in dataloader:
    optimizer.zero_grad()
    
    with autocast():
        output = model(data)
        loss = criterion(output, target)
    
    scaler.scale(loss).backward()
    
    # 可選：梯度裁剪
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    scaler.step(optimizer)
    scaler.update()
```

### 進階閱讀

- [PyTorch AMP 官方文檔](https://pytorch.org/docs/stable/amp.html)
- [NVIDIA Mixed Precision Training](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/)
- [BFloat16 論文](https://arxiv.org/abs/1905.12322)

## 練習

1. 在你自己的模型上實現混合精度訓練，比較性能提升
2. 嘗試不同的梯度縮放初始值，觀察對訓練穩定性的影響
3. 比較 FP16 和 BF16 在大模型上的性能差異（如果有 Ampere GPU）
4. 實現自定義的混合精度策略，某些層用 FP32，其他層用 FP16
5. 使用 Profiler 分析混合精度訓練的性能瓶頸